In [2]:
import pandas as pd
import numpy as np

In [4]:
customer_features = pd.read_csv( "../data/processed/customer_features.csv")

customer_features.head()

,user_id,total_orders,total_spend,total_quantity,average_order_value,first_order_date,last_order_date,age,gender,marital_status,occupation
0,1,2,1672,5,836.000000,2017-12-12,2018-11-14,20,Female,Single,Student
1,2,3,4061,7,1353.666667,2018-02-13,2018-08-03,24,Female,Single,Student
2,3,1,21542,500,21542.000000,2019-05-15,2019-05-15,22,Male,Single,Student
3,4,1,102,1,102.000000,2019-02-06,2019-02-06,22,Female,Single,Student
4,5,2,52047,50,26023.500000,2018-03-02,2018-07-25,22,Male,Single,Student


CONVERT DATES

In [5]:
customer_features["first_order_date"] = pd.to_datetime(customer_features["first_order_date"])

customer_features["last_order_date"] = pd.to_datetime(customer_features["last_order_date"])

FINDING ANALYSIS DATE

In [6]:
analysis_date = customer_features["last_order_date"].max()

print("Analysis Date:", analysis_date)

Analysis Date: 2020-06-26 00:00:00


CALCULATE RECENCY

In [7]:
customer_features["recency"] = (analysis_date - customer_features["last_order_date"]).dt.days

CALCULATE FREQUENCY

In [8]:
customer_features["frequency"] = customer_features["total_orders"]

CALCULATE MONETARY

In [9]:
customer_features["monetary"] = customer_features["total_spend"]

In [10]:
customer_features[["user_id", "recency", "frequency", "monetary"]].head(10)

,user_id,recency,frequency,monetary
0,1,590,2,1672
1,2,693,3,4061
2,3,408,1,21542
3,4,506,1,102
4,5,702,2,52047
5,6,623,1,83
6,8,371,1,241
7,9,232,3,7940
8,10,67,2,2773
9,13,147,3,3819


RECENCY SCORE

In [11]:
customer_features["R_score"] = pd.qcut(customer_features["recency"],5,labels=[5, 4, 3, 2, 1])

FREQUENCY SCORE

In [12]:
customer_features["F_score"] = pd.cut(customer_features["frequency"],bins=[0, 1, 2, 3, 4, float("inf")],
    labels=[1, 2, 3, 4, 5])

MONETARY SCORE

In [13]:
customer_features["M_score"] = pd.qcut(customer_features["monetary"].rank(method="first"),5,
                                       labels=[1, 2, 3, 4, 5])

CONVERTING SCORE TO INTEGERS

In [14]:
customer_features["R_score"] = customer_features["R_score"].astype(int)
customer_features["F_score"] = customer_features["F_score"].astype(int)
customer_features["M_score"] = customer_features["M_score"].astype(int)

CREATING OVERALL RFM SCORE

In [15]:
customer_features["RFM_score"] = (customer_features["R_score"].astype(str) +
                                  customer_features["F_score"].astype(str) +
                                  customer_features["M_score"].astype(str))

CREATING NUMERICAL RFM MODEL

In [16]:
customer_features["RFM_total"] = (
    customer_features["R_score"] +
    customer_features["F_score"] +
    customer_features["M_score"]
)

In [17]:
customer_features[["user_id","recency","frequency","monetary","R_score","F_score","M_score","RFM_score",
                   "RFM_total"]].head(10)

,user_id,recency,frequency,monetary,R_score,F_score,M_score,RFM_score,RFM_total
0,1,590,2,1672,2,2,3,223,7
1,2,693,3,4061,1,3,4,134,8
2,3,408,1,21542,3,1,5,315,9
3,4,506,1,102,2,1,1,211,4
4,5,702,2,52047,1,2,5,125,8
5,6,623,1,83,2,1,1,211,4
6,8,371,1,241,3,1,1,311,5
7,9,232,3,7940,4,3,4,434,11
8,10,67,2,2773,5,2,3,523,10
9,13,147,3,3819,4,3,4,434,11


CHECKING SCORE DISTRIBUTION

In [18]:
print(customer_features["R_score"].value_counts().sort_index())
print(customer_features["F_score"].value_counts().sort_index())
print(customer_features["M_score"].value_counts().sort_index())

R_score
1    15462
2    15531
3    15450
4    15440
5    15663
Name: count, dtype: int64
F_score
1    33660
2    25321
3    12280
4     4556
5     1729
Name: count, dtype: int64
M_score
1    15510
2    15509
3    15509
4    15509
5    15509
Name: count, dtype: int64


CREATING CUSTOMER SEGMENTS

In [19]:
def rfm_segment(row):
    r = row["R_score"]
    f = row["F_score"]
    m = row["M_score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"

    elif r >= 3 and f >= 4:
        return "Loyal Customers"

    elif r >= 4 and f <= 2:
        return "New / Promising"

    elif r <= 2 and f >= 4:
        return "At Risk - High Frequency"

    elif r <= 2 and m >= 4:
        return "At Risk - High Value"

    elif r <= 2 and f <= 2 and m <= 2:
        return "Lost / Low Value"

    else:
        return "Potential Loyalists"


customer_features["rfm_segment"] = customer_features.apply(
    rfm_segment,
    axis=1
)

NUMBER OF CUSTOMERS IN EACH SEGMEMT

In [20]:
customer_features["rfm_segment"].value_counts()

rfm_segment
Potential Loyalists         27252
New / Promising             19513
Lost / Low Value            15211
At Risk - High Value         9285
Champions                    3530
Loyal Customers              2139
At Risk - High Frequency      616
Name: count, dtype: int64

CREATING SEGMENT SUMMARY

In [21]:
segment_summary = (customer_features.groupby("rfm_segment").agg(customers=("user_id", "count"),
        avg_recency=("recency", "mean"),avg_frequency=("frequency", "mean"),avg_monetary=("monetary", "mean"))
    .sort_values("customers", ascending=False).reset_index())

segment_summary

,rfm_segment,customers,avg_recency,avg_frequency,avg_monetary
0,Potential Loyalists,27252,389.135660,2.077169,11476.546529
1,New / Promising,19513,151.391073,1.567878,10199.373392
2,Lost / Low Value,15211,720.294129,1.196831,342.777069
3,At Risk - High Value,9285,684.068282,1.694777,28612.285407
4,Champions,3530,129.621530,4.445042,37314.858357
5,Loyal Customers,2139,269.742403,4.254792,16525.288920
6,At Risk - High Frequency,616,577.407468,4.211039,24320.823052


In [22]:
import os

os.makedirs("../outputs/tables", exist_ok=True)

In [23]:
segment_summary.to_csv("../outputs/tables/customer_rfm.csv",index=False)

In [24]:
customer_features.to_csv("../data/processed/customer_rfm.csv",index=False)